# RL Portfolio Trader - Evaluation Notebook

This notebook demonstrates:
1. Loading trained model
2. Running backtests (2018-2025)
3. Comparing against benchmarks (equal-weight, Markowitz)
4. Visualizing efficient frontier
5. Performance metrics analysis

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Add src to path
sys.path.append('../src')

from trading_env import TradingEnvironment
from ddpg_agent import DDPGAgent
from backtester import Backtester

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 8)

%load_ext autoreload
%autoreload 2

## 1. Setup Environment and Load Model

In [ ]:
# Define assets
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'JPM', 'BAC', 'JNJ', 'PG', 'GBTC', 'ETHE']

# Create environment for full backtest period (2018-2025)
env = TradingEnvironment(
    tickers=tickers,
    start_date='2018-01-01',
    end_date='2025-01-01',
    initial_balance=100000,
    transaction_cost=0.001,
    lookback_window=30,
    vix_window=20,
    risk_penalty=0.5
)

print(f"Environment created for period: 2018-2025")
print(f"Assets: {tickers}")
print(f"Total trading days: {len(env.data)}")

In [ ]:
# Load trained agent
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

agent = DDPGAgent(
    state_dim=state_dim,
    action_dim=action_dim,
    hidden_dims=[256, 256]
)

# Load weights
model_path = '../results/final_model.pt'
if os.path.exists(model_path):
    agent.load(model_path)
    print(f"✓ Model loaded from {model_path}")
else:
    print(f"✗ Model not found at {model_path}")
    print("Please train the model first using train.py")

## 2. Run Backtests

In [ ]:
# Create backtester
backtester = Backtester(env, agent)

# Run RL agent backtest
rl_results = backtester.run_backtest("RL Agent")
print("\nRL Agent Performance:")
for key, value in rl_results['metrics'].items():
    print(f"  {key}: {value:.4f}")

In [ ]:
# Run equal-weight benchmark
eq_results = backtester.run_equal_weight_benchmark()
print("\nEqual Weight Performance:")
for key, value in eq_results['metrics'].items():
    print(f"  {key}: {value:.4f}")

In [ ]:
# Run Markowitz benchmark
mv_results = backtester.run_markowitz_benchmark()
print("\nMarkowitz Optimal Performance:")
for key, value in mv_results['metrics'].items():
    print(f"  {key}: {value:.4f}")
print(f"\nOptimal Weights:")
for ticker, weight in zip(tickers, mv_results['optimal_weights']):
    print(f"  {ticker}: {weight*100:.2f}%")

## 3. Performance Comparison

In [ ]:
# Generate performance report
report = backtester.generate_performance_report()
print("\nPerformance Comparison:")
print(report.to_string())

In [ ]:
# Plot performance comparison
backtester.plot_performance_comparison(save_path='../results/performance_comparison.html')
print("Performance comparison saved to ../results/performance_comparison.html")

## 4. Efficient Frontier Visualization

In [ ]:
# Plot efficient frontier
backtester.plot_efficient_frontier(
    n_portfolios=5000,
    save_path='../results/efficient_frontier.html'
)
print("Efficient frontier saved to ../results/efficient_frontier.html")

## 5. Portfolio Weights Evolution

In [ ]:
# Plot weights evolution
backtester.plot_weights_evolution(save_path='../results/weights_evolution.png')
print("Weights evolution saved to ../results/weights_evolution.png")

## 6. Detailed Analysis

In [ ]:
# Analyze rolling Sharpe ratio
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

for name, result in backtester.results.items():
    returns = pd.Series(result['daily_returns'])
    rolling_sharpe = returns.rolling(252).mean() / returns.rolling(252).std() * np.sqrt(252)
    
    axes[0].plot(rolling_sharpe, label=name, linewidth=2)

axes[0].axhline(y=1.5, color='r', linestyle='--', label='Target (1.5)')
axes[0].set_title('Rolling Sharpe Ratio (1-Year Window)')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Sharpe Ratio')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Analyze rolling volatility
for name, result in backtester.results.items():
    returns = pd.Series(result['daily_returns'])
    rolling_vol = returns.rolling(252).std() * np.sqrt(252)
    
    axes[1].plot(rolling_vol, label=name, linewidth=2)

axes[1].set_title('Rolling Volatility (1-Year Window)')
axes[1].set_xlabel('Days')
axes[1].set_ylabel('Volatility')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/rolling_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Analyze average weights for RL agent
rl_weights = np.array(rl_results['weights_history'])
avg_weights = rl_weights.mean(axis=0)
std_weights = rl_weights.std(axis=0)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(tickers))
ax.bar(x, avg_weights, yerr=std_weights, capsize=5, alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(tickers, rotation=45)
ax.set_ylabel('Weight')
ax.set_title('RL Agent - Average Portfolio Weights (±1 std)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/average_weights.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nAverage Weights:")
for ticker, weight, std in zip(tickers, avg_weights, std_weights):
    print(f"  {ticker}: {weight*100:.2f}% (±{std*100:.2f}%)")

## 7. Risk Analysis

In [ ]:
# Calculate Value at Risk (VaR) and Conditional VaR
def calculate_var_cvar(returns, confidence=0.95):
    var = np.percentile(returns, (1 - confidence) * 100)
    cvar = returns[returns <= var].mean()
    return var, cvar

print("\nRisk Metrics (95% confidence):")
print("="*60)
for name, result in backtester.results.items():
    returns = np.array(result['daily_returns'])
    var, cvar = calculate_var_cvar(returns)
    print(f"\n{name}:")
    print(f"  Daily VaR (95%): {var*100:.4f}%")
    print(f"  Daily CVaR (95%): {cvar*100:.4f}%")
    print(f"  Annual VaR (95%): {var*np.sqrt(252)*100:.4f}%")
    print(f"  Annual CVaR (95%): {cvar*np.sqrt(252)*100:.4f}%")

## 8. Summary and Conclusions

In [ ]:
# Check if objectives are met
rl_metrics = rl_results['metrics']

print("\n" + "="*80)
print("OBJECTIVE ACHIEVEMENT")
print("="*80)

sharpe_target = 1.5
drawdown_target = 0.15

sharpe_achieved = rl_metrics['sharpe_ratio'] > sharpe_target
drawdown_achieved = abs(rl_metrics['max_drawdown']) < drawdown_target

print(f"\nTarget: Sharpe Ratio > {sharpe_target}")
print(f"  Achieved: {rl_metrics['sharpe_ratio']:.4f}")
print(f"  Status: {'✓ PASS' if sharpe_achieved else '✗ FAIL'}")

print(f"\nTarget: Max Drawdown < {drawdown_target*100}%")
print(f"  Achieved: {abs(rl_metrics['max_drawdown'])*100:.2f}%")
print(f"  Status: {'✓ PASS' if drawdown_achieved else '✗ FAIL'}")

print(f"\nOverall: {'✓ ALL OBJECTIVES MET' if (sharpe_achieved and drawdown_achieved) else '✗ SOME OBJECTIVES NOT MET'}")

# Compare to benchmarks
print("\n" + "="*80)
print("BENCHMARK COMPARISON")
print("="*80)

for benchmark_name in ['Equal Weight', 'Markowitz Optimal']:
    if benchmark_name in backtester.results:
        benchmark_sharpe = backtester.results[benchmark_name]['metrics']['sharpe_ratio']
        improvement = ((rl_metrics['sharpe_ratio'] - benchmark_sharpe) / abs(benchmark_sharpe)) * 100
        print(f"\nRL Agent vs {benchmark_name}:")
        print(f"  Sharpe improvement: {improvement:+.2f}%")
        print(f"  Status: {'✓ Outperformed' if improvement > 0 else '✗ Underperformed'}")

In [ ]:
# Save final report
with open('../results/evaluation_summary.txt', 'w') as f:
    f.write("RL PORTFOLIO TRADER - EVALUATION SUMMARY\n")
    f.write("="*80 + "\n\n")
    f.write(f"Backtest Period: 2018-2025 (7 years)\n")
    f.write(f"Assets: {', '.join(tickers)}\n\n")
    f.write(report.to_string())
    f.write("\n\n")
    f.write(f"Objectives Achievement:\n")
    f.write(f"  Sharpe Ratio > 1.5: {'PASS' if sharpe_achieved else 'FAIL'} ({rl_metrics['sharpe_ratio']:.4f})\n")
    f.write(f"  Max Drawdown < 15%: {'PASS' if drawdown_achieved else 'FAIL'} ({abs(rl_metrics['max_drawdown'])*100:.2f}%)\n")

print("\nEvaluation summary saved to ../results/evaluation_summary.txt")